KG-06 is the natural transition from our Python-built graph to a real graph database: Neo4j + Cypher.
# CELL 01

# Knowledge Graph — Notebook 6
# Building a Travel Planning Knowledge Graph with Neo4j and Cypher

## Running Problem

We have been building our Travel Knowledge Graph
using Python data structures.

Now we move to a real graph database:

> Neo4j

And we will use:

> Cypher

as the graph query language.

The travel problem remains the same.

The knowledge remains the same.

Only the implementation changes.

---

## C2C Journey

Real-world travel problem
        ↓
Knowledge
        ↓
Triples
        ↓
Graph representation
        ↓
Python implementation
        ↓
Neo4j graph database
        ↓
Cypher queries
        ↓
Traversal
        ↓
Multi-hop reasoning

---

## Main Goal

By the end of this notebook, students should understand:

1. Why a graph database is useful
2. How our Python graph maps to Neo4j
3. Nodes and relationships in Neo4j
4. Properties on nodes and relationships
5. Cypher syntax
6. Pattern matching
7. Filtering
8. Multi-hop traversal
9. Aggregation
10. Path-based travel queries

The goal is NOT to memorize Neo4j commands.

The goal is to understand:

> How a Knowledge Graph becomes a real graph database.

# CELL 02

# Recall the Travel Knowledge Graph

Our travel knowledge contains facts such as:

Chennai is located in Tamil Nadu.

Mahabalipuram is near Chennai.

Mahabalipuram is a heritage destination.

Hotel SeaView is located in Mahabalipuram.

Hotel SeaView costs ₹3500 per night.

Chennai is connected to Bengaluru.

Bengaluru is connected to Mysuru.

---

We already represented these facts as triples:

    (Mahabalipuram, NEAR, Chennai)

    (Mahabalipuram, HAS_CATEGORY, Heritage)

    (Hotel SeaView, LOCATED_IN, Mahabalipuram)

    (Hotel SeaView, PRICE_PER_NIGHT, 3500)

Now we will represent the same knowledge
inside Neo4j.

# CELL 03

# From Python Graph to Graph Database

In earlier notebooks we created something like:

    graph = {
        "Chennai": [
            ("CONNECTED_TO", "Bengaluru")
        ]
    }

This was an in-memory Python representation.

It is useful for learning.

But a real application needs a system that can:

- store large graphs
- persist data
- query efficiently
- manage relationships
- support multiple users/applications

That is where a graph database becomes useful.

Today:

    Python graph
         ↓
    Neo4j graph database

# CELL 04

# What Is Neo4j?

Neo4j is a graph database.

Instead of primarily thinking in terms of:

    Tables
    Rows
    Columns

we think in terms of:

    Nodes
    Relationships
    Properties

For our travel problem:

    Node
       ↓
    Chennai

    Relationship
       ↓
    CONNECTED_TO

    Node
       ↓
    Bengaluru

The graph structure is stored directly.

# CELL 05


# Neo4j and Our Knowledge Graph

Our conceptual model:

    Entity → Relationship → Entity

Neo4j represents this as:

    (Node)-[Relationship]->(Node)

For example:

    (Chennai)-[:CONNECTED_TO]->(Bengaluru)

Another example:

    (HotelSeaView)-[:LOCATED_IN]->(Mahabalipuram)

This is very close to the triple representation
we have already learned.

# CELL 06

# Neo4j Installation / Access

For this notebook we will use Neo4j.

There are two common ways to work with Neo4j Aura:





The exact installation environment can be
configured separately.

Neo4j Setup

For this practical session, we will use Neo4j AuraDB Free, Neo4j's cloud-hosted graph database.

Create a Neo4j account.
Open the Neo4j Aura Console.
Create an AuraDB Free instance.
Choose a blank database.
Save the database username and password safely.
Wait until the database status is RUNNING.
Open the Neo4j Browser/query interface.
We will execute the Cypher queries from this notebook in Neo4j.

Important: The Python code in this notebook is instructional where applicable. The Cypher queries are executed in Neo4j Browser, not in the Python/Jupyter kernel.

Note:AuraDB Free database automatically pauses after three days of inactivity, with email notification beforehand. It can be resumed, and the data is not lost.

create a blank database, rather than starting with the existing  Movie database.

Neo4j's learning resources also provide a Sandbox with pre-built datasets, 
which is excellent for learning Neo4j itself.

But for  KG-06, we don't want the Movie database.

We want:

Our own Travel Planning Knowledge Graph.

# CELL 07

# Neo4j Browser

Neo4j Browser provides an interactive environment
where we can:

- create graph data
- execute Cypher
- inspect results
- visualize nodes and relationships

The typical classroom workflow is:

    Open Neo4j Browser
          ↓
    Enter Cypher
          ↓
    Execute
          ↓
    Inspect graph
          ↓
    Ask another question

This makes the graph visible to students.

# CELL 08

# First Cypher Statement

Before creating the travel graph,
let us understand the basic syntax.

Cypher uses patterns.

Example:

    (c:City {name: "Chennai"})

Here:

    c
    ↓
    variable

    City
    ↓
    label

    name
    ↓
    property

    Chennai
    ↓
    property value

# CELL 09

# Node Representation

A Neo4j node can be represented as:

    (variable:Label {property: value})

Example:

    (c:City {name: "Chennai"})

This means:

- variable = c
- label = City
- property = name
- value = Chennai

The label helps classify the node.

The property stores information about the node.

# CELL 10

# Relationship Representation

A relationship is represented as:

    (node)-[:RELATIONSHIP]->(node)

Example:

    (chennai)-[:CONNECTED_TO]->(bengaluru)

The relationship type is:

    CONNECTED_TO

The direction is:

    Chennai → Bengaluru

This is different from merely storing
two values in a table.

The relationship itself is part of the graph.

# CELL 11

# Our First Travel Graph

Let us create:

    Chennai → Bengaluru

The Cypher pattern is:

    (Chennai)-[:CONNECTED_TO]->(Bengaluru)

But Neo4j needs actual nodes and relationships.

We will therefore use CREATE.


In [1]:
#cell 12
%pip install neo4j

Note: you may need to restart the kernel to use updated packages.


In [2]:
#cell 13
from neo4j import GraphDatabase

URI = "neo4j+s://XXXXXX.databases.neo4j.io"
USERNAME = "XXXXXX"
PASSWORD = "XXXXXXXXX"

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected to Neo4j successfully!")

Connected to Neo4j successfully!


In [3]:
# CELL 14

query = """
MATCH (n)
DETACH DELETE n
"""

with driver.session() as session:
    session.run(query)

print("Database cleared.")

Database cleared.


In [4]:
# CELL 12

query = """
CREATE (c:City {name: "Chennai"})
CREATE (b:City {name: "Bengaluru"})
CREATE (c)-[:CONNECTED_TO]->(b)
"""

with driver.session() as session:
    session.run(query)

print("Travel graph created successfully.")

Travel graph created successfully.


# CELL 13

# Inspect the Graph

Now ask Neo4j:

> Show me the travel graph.

Cypher:

    MATCH (n)
    RETURN n

MATCH is used to find patterns.

RETURN specifies what we want to see.


In [5]:

    # CELL 14

query = """
MATCH (n)
RETURN n
"""

with driver.session() as session:
    result = session.run(query)

    for record in result:
        print(record)

<Record n=<Node element_id='4:07668400-8997-45bb-b601-5170a21901bd:1' labels=frozenset({'City'}) properties={'name': 'Chennai'}>>
<Record n=<Node element_id='4:07668400-8997-45bb-b601-5170a21901bd:2' labels=frozenset({'City'}) properties={'name': 'Bengaluru'}>>


In [6]:
!pip install pyvis

Defaulting to user installation because normal site-packages is not writeable


In [7]:
import sys
import subprocess

# Force installation directly into the current Jupyter kernel environment
subprocess.check_call([sys.executable, "-m", "pip", "install", "pyvis"])

# Reload path to ensure Jupyter recognizes the new package
import sys
if sys.path[0] != "":
    sys.path.insert(0, "")

# Test import
from pyvis.network import Network
from IPython.display import HTML
print("Pyvis successfully loaded!")

Pyvis successfully loaded!


In [10]:
# CELL 18

query = """
MERGE (chennai:City {name: "Chennai"})
MERGE (bengaluru:City {name: "Bengaluru"})
MERGE (mysuru:City {name: "Mysuru"})
MERGE (tamilnadu:State {name: "Tamil Nadu"})

MERGE (mahabalipuram:Destination {name: "Mahabalipuram"})
MERGE (pondicherry:Destination {name: "Pondicherry"})

MERGE (heritage1:Category {name: "Heritage"})

MERGE (hotel1:Hotel {name: "Hotel SeaView", price: 3500})
MERGE (hotel2:Hotel {name: "Hotel Heritage", price: 3000})
"""

with driver.session() as session:
    session.run(query)
    print("work done")

# CREATE
# (chennai:City {name: "Chennai"}),
# (bengaluru:City {name: "Bengaluru"}),
# (mysuru:City {name: "Mysuru"}),
# (tamilnadu:State {name: "Tamil Nadu"}),

# (mahabalipuram:Destination {name: "Mahabalipuram"}),
# (pondicherry:Destination {name: "Pondicherry"}),

# (heritage1:Category {name: "Heritage"}),

# (hotel1:Hotel {name: "Hotel SeaView", price: 3500}),
# (hotel2:Hotel {name: "Hotel Heritage", price: 3000})

work done


# CELL 19

# Create Travel Relationships

Now connect the nodes.

The graph will contain relationships such as:

Chennai
   ↓ LOCATED_IN
Tamil Nadu

Mahabalipuram
   ↓ NEAR
Chennai

Mahabalipuram
   ↓ HAS_CATEGORY
Heritage

Hotel SeaView
   ↓ LOCATED_IN
Mahabalipuram

Hotel SeaView
   ↓ PRICE
3500

Chennai
   ↓ CONNECTED_TO
Bengaluru
   ↓ CONNECTED_TO
Mysuru
Note:

In Neo4j we can choose whether a value such as
price should be a property or a separate node.

For structured attributes such as price,
a node property is usually more convenient.
------
<!-- It connects your existing nodes with 9 directed relationships:Location hierarchy: Chennai $\rightarrow$ Tamil Nadu, Hotel SeaView $\rightarrow$ Mahabalipuram, Hotel Heritage $\rightarrow$ PondicherryProximity & Travel links: Mahabalipuram $\rightarrow$ Chennai, Pondicherry $\rightarrow$ Chennai, Chennai $\rightarrow$ Bengaluru, Bengaluru $\rightarrow$ MysuruCategorization: Mahabalipuram $\rightarrow$ Heritage, Pondicherry $\rightarrow$ Heritage -->

In [11]:

## Cell 20
query = """
MATCH (c:City {name: "Chennai"})
MATCH (b:City {name: "Bengaluru"})
MATCH (m:City {name: "Mysuru"})
MATCH (t:State {name: "Tamil Nadu"})
MATCH (ma:Destination {name: "Mahabalipuram"})
MATCH (p:Destination {name: "Pondicherry"})
MATCH (h:Category {name: "Heritage"})
MATCH (hs:Hotel {name: "Hotel SeaView"})
MATCH (hh:Hotel {name: "Hotel Heritage"})

MERGE (c)-[:LOCATED_IN]->(t)
MERGE (ma)-[:NEAR]->(c)
MERGE (p)-[:NEAR]->(c)
MERGE (ma)-[:HAS_CATEGORY]->(h)
MERGE (p)-[:HAS_CATEGORY]->(h)
MERGE (hs)-[:LOCATED_IN]->(ma)
MERGE (hh)-[:LOCATED_IN]->(p)
MERGE (c)-[:CONNECTED_TO]->(b)
MERGE (b)-[:CONNECTED_TO]->(m)
"""

with driver.session() as session:
    session.run(query)
    print("Relationships merged safely without creating new duplicates.")

#----duplicate will be there
# query = """
# MATCH (c:City {name: "Chennai"})
# MATCH (b:City {name: "Bengaluru"})
# MATCH (m:City {name: "Mysuru"})
# MATCH (t:State {name: "Tamil Nadu"})
# MATCH (ma:Destination {name: "Mahabalipuram"})
# MATCH (p:Destination {name: "Pondicherry"})
# MATCH (h:Category {name: "Heritage"})
# MATCH (hs:Hotel {name: "Hotel SeaView"})
# MATCH (hh:Hotel {name: "Hotel Heritage"})

# CREATE
# (c)-[:LOCATED_IN]->(t),
# (ma)-[:NEAR]->(c),
# (p)-[:NEAR]->(c),
# (ma)-[:HAS_CATEGORY]->(h),
# (p)-[:HAS_CATEGORY]->(h),
# (hs)-[:LOCATED_IN]->(ma),
# (hh)-[:LOCATED_IN]->(p),
# (c)-[:CONNECTED_TO]->(b),
# (b)-[:CONNECTED_TO]->(m)
# """

# with driver.session() as session:
#     session.run(query)
#     print("work done")


# MATCH (c:City {name: "Chennai"})
# MATCH (b:City {name: "Bengaluru"})
# MATCH (m:City {name: "Mysuru"})
# MATCH (t:State {name: "Tamil Nadu"})
# MATCH (ma:Destination {name: "Mahabalipuram"})
# MATCH (p:Destination {name: "Pondicherry"})
# MATCH (h:Category {name: "Heritage"})
# MATCH (hs:Hotel {name: "Hotel SeaView"})
# MATCH (hh:Hotel {name: "Hotel Heritage"})

# CREATE
# (c)-[:LOCATED_IN]->(t),
# (ma)-[:NEAR]->(c),
# (p)-[:NEAR]->(c),
# (ma)-[:HAS_CATEGORY]->(h),
# (p)-[:HAS_CATEGORY]->(h),
# (hs)-[:LOCATED_IN]->(ma),
# (hh)-[:LOCATED_IN]->(p),
# (c)-[:CONNECTED_TO]->(b),
# (b)-[:CONNECTED_TO]->(m)

Relationships merged safely without creating new duplicates.


# CELL 21

# Inspect the Complete Travel Graph

Now we ask Neo4j to return
the graph structure.

This is where Neo4j Browser becomes
particularly useful.

We are not just printing rows.

We can visualize the graph.


In [12]:
# CELL 22
# Initialize Pyvis Network using already loaded modules
net = Network(notebook=True, cdn_resources='remote', height="600px", width="100%", directed=True)

query = """
MATCH (n)-[r]->(m)
RETURN n, r, m
"""

with driver.session() as session:
    result = session.run(query)
    
    for record in result:
        node_from = record["n"]
        node_to = record["m"]
        rel = record["r"]
        
        from_id = str(node_from.element_id)
        to_id = str(node_to.element_id)
        
        from_label = node_from.get("name", list(node_from.labels)[0])
        to_label = node_to.get("name", list(node_to.labels)[0])
        
        net.add_node(from_id, label=str(from_label), title=str(dict(node_from)))
        net.add_node(to_id, label=str(to_label), title=str(dict(node_to)))
        net.add_edge(from_id, to_id, label=rel.type)

net.write_html("graph.html")
IFrame(src="graph.html", width="100%", height="600px")

# MATCH (n)-[r]->(m)
# RETURN n, r, m

# CELL 23

# A First Important Observation

Compare our Python graph with Neo4j.

Python:

    graph["Chennai"]

Neo4j:

    MATCH (c:City {name: "Chennai"})-[r]->(x)
    RETURN r, x

The conceptual question is the same:

> What is connected to Chennai?

The implementation has changed.

This is the important transition:

    CONCEPT
       ↓
    PYTHON GRAPH
       ↓
    GRAPH DATABASE

# CELL 24

# Query 1 — Where Is Chennai Located?

Our question:

> Where is Chennai located?

Graph pattern:

    Chennai
       ↓
    LOCATED_IN
       ↓
    ?

Cypher pattern:

    (Chennai)-[:LOCATED_IN]->(location)

In [13]:
# CELL 25
query = """
MATCH (c:City {name: "Chennai"})-[:LOCATED_IN]->(location)
RETURN location
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        loc_node = record["location"]
        print("Chennai is located in:", loc_node.get("name", list(loc_node.labels)[0]))

# MATCH (c:City {name: "Chennai"})
#       -[:LOCATED_IN]->
#       (location)

# RETURN location

Chennai is located in: Tamil Nadu


# CELL 26

# Query 2 — Which Places Are Near Chennai?

Question:

> Which places are near Chennai?

Graph pattern:

    ?
    ↓
    NEAR
    ↓
    Chennai

Cypher:

    (place)-[:NEAR]->(Chennai)

In [14]:
# CELL 27
query = """
MATCH (place)-[:NEAR]->(c:City {name: "Chennai"})
RETURN place
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        place_node = record["place"]
        print("Near Chennai:", place_node.get("name", list(place_node.labels)[0]))

# MATCH (place)-[:NEAR]->(c:City {name: "Chennai"})
# RETURN place

Near Chennai: Mahabalipuram
Near Chennai: Pondicherry


# CELL 28

# Query 3 — Which Cities Are Connected to Chennai?

Question:

> Which cities are directly connected to Chennai?

Graph pattern:

    Chennai
       ↓
    CONNECTED_TO
       ↓
       ?


In [15]:
# CELL 29 which cities are connected to chennai?
query = """
MATCH (c:City {name: "Chennai"})-[:CONNECTED_TO]->(city)
RETURN city
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        city_node = record["city"]
        print("Chennai is connected to:", city_node.get("name", list(city_node.labels)[0]))
# MATCH (c:City {name: "Chennai"})
#       -[:CONNECTED_TO]->
#       (city)

# RETURN city

Chennai is connected to: Bengaluru


# CELL 30

# Cypher Pattern Matching

We can now see the central idea of Cypher.

Instead of thinking:

    "Which table should I JOIN?"

we think:

    "What graph pattern am I looking for?"

For example:

    (place)-[:NEAR]->(chennai)

or:

    (hotel)-[:LOCATED_IN]->(destination)

Cypher matches the pattern against
the graph.

# CELL 31

# Query 4 — Find Hotels

Question:

> Which hotels are in Mahabalipuram?

Pattern:

    Hotel
      ↓
    LOCATED_IN
      ↓
    Mahabalipuram


In [16]:
# CELL 32-which hotels are in Mahabalipuram-It filters for nodes with the 
# label Hotel that have a LOCATED_IN relationship pointing to the Destination 
# node Mahabalipuram and returns the matching hotel node (Hotel SeaView).

query = """
MATCH (hotel:Hotel)-[:LOCATED_IN]->(destination:Destination {name: "Mahabalipuram"})
RETURN hotel
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        hotel_node = record["hotel"]
        hotel_name = hotel_node.get("name", list(hotel_node.labels)[0])
        hotel_price = hotel_node.get("price", "N/A")
        print(f"Hotel in Mahabalipuram: {hotel_name} (Price: ₹{hotel_price})")
# MATCH (hotel:Hotel)
#       -[:LOCATED_IN]->
#       (destination:Destination {name: "Mahabalipuram"})

# RETURN hotel

Hotel in Mahabalipuram: Hotel SeaView (Price: ₹3500)


# CELL 33

# Query 5 — Find Affordable Hotels

Question:

> Which hotels cost less than ₹3500?

This time we combine:

1. Graph pattern
2. Property filtering

The graph tells us:

    Hotel → LOCATED_IN → Destination

The property tells us:

    hotel.price < 3500

In [17]:
# CELL 34 To find affordable Hotels
# It scans all nodes labeled Hotel, applies a condition to filter for price < 3500, and returns the matching hotel name and
# price (Hotel Heritage at ₹3000).
query = """
MATCH (hotel:Hotel)
WHERE hotel.price < 3500
RETURN hotel.name AS name, hotel.price AS price
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        print(f"Hotel: {record['name']} | Price: ₹{record['price']}")
# MATCH (hotel:Hotel)
# WHERE hotel.price < 3500
# RETURN hotel.name, hotel.price

Hotel: Hotel Heritage | Price: ₹3000


# CELL 35

# Important Observation

Neo4j can combine:

    Relationships
        +
    Properties
        +
    Conditions

For example:

    Hotel
      ↓
    LOCATED_IN
      ↓
    Destination

and:

    hotel.price < 3500

This is important because real-world questions
often combine relationships and properties.

# CELL 36

# Query 6 — Heritage Destinations

Question:

> Which destinations are heritage destinations?

Graph pattern:

    Destination
          ↓
    HAS_CATEGORY
          ↓
       Heritage


In [18]:
# CELL 37 Which destinations are heritage destinations?
# It searches for all nodes with the label Destination that have an outgoing
# HAS_CATEGORY relationship to the Category node Heritage and 
# returns them (Mahabalipuram and Pondicherry).
query = """
MATCH (destination:Destination)-[:HAS_CATEGORY]->(category:Category {name: "Heritage"})
RETURN destination
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        dest_node = record["destination"]
        print("Heritage Destination:", dest_node.get("name", list(dest_node.labels)[0]))

# MATCH (destination:Destination)
#       -[:HAS_CATEGORY]->
#       (category:Category {name: "Heritage"})

# RETURN destination

Heritage Destination: Mahabalipuram
Heritage Destination: Pondicherry


# CELL 38

# Query 7 — Heritage Destinations Near Chennai

Now combine two relationships.

Question:

> Which heritage destinations are near Chennai?

Reasoning:

    Destination
        ↓
    HAS_CATEGORY
        ↓
    Heritage

AND

    Destination
        ↓
    NEAR
        ↓
    Chennai

This is a graph-pattern question.

In [19]:
# CELL 39-what are the heritage destinations near Chennai-multi-criteria query and print all Heritage destinations 
#located near Chennai:
# It intersects two conditions simultaneously:

#     The Destination must have a HAS_CATEGORY link to the Heritage category.

#     The same Destination must also have a NEAR link to Chennai.

# It returns both Mahabalipuram and Pondicherry.

query = """
MATCH (destination:Destination)-[:HAS_CATEGORY]->(category:Category {name: "Heritage"}),
      (destination)-[:NEAR]->(chennai:City {name: "Chennai"})
RETURN destination
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        dest_node = record["destination"]
        print("Heritage Destination near Chennai:", dest_node.get("name", list(dest_node.labels)[0]))
# MATCH (destination:Destination)
#       -[:HAS_CATEGORY]->
#       (category:Category {name: "Heritage"}),
#       (destination)-[:NEAR]->
#       (chennai:City {name: "Chennai"})

# RETURN destination

Heritage Destination near Chennai: Mahabalipuram
Heritage Destination near Chennai: Pondicherry


# CELL 40

# Multi-Hop Traversal

Now we return to an important concept
from KG-03.

Question:

> Can we travel from Chennai to Mysuru
> through connected cities?

We have:

    Chennai
       ↓
    Bengaluru
       ↓
    Mysuru

This is a two-hop path.

In [20]:
# CELL 41
# It performs a multi-hop path traversal using -[:CONNECTED_TO*1..2]-> to match routes between
# Chennai and Mysuru that are between 1 and 2 hops away.

# It traverses through Bengaluru (Chennai -> Bengaluru -> Mysuru) and 
# returns the entire path graph object.
query = """
MATCH path = (c:City {name: "Chennai"})-[:CONNECTED_TO*1..2]->(m:City {name: "Mysuru"})
RETURN path
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        path = record["path"]
        
        # Extract node names along the path sequence
        nodes_in_path = [node.get("name", list(node.labels)[0]) for node in path.nodes]
        
        # Display as a connected route
        route_str = " -> ".join(nodes_in_path)
        print("Route found:", route_str)

# MATCH path =
#     (c:City {name: "Chennai"})
#     -[:CONNECTED_TO*1..2]->
#     (m:City {name: "Mysuru"})

# RETURN path

Route found: Chennai -> Bengaluru -> Mysuru


# CELL 42

# Understanding *1..2

In Cypher:

    [:CONNECTED_TO*1..2]

means:

    one or two CONNECTED_TO relationships

So Neo4j can search:

    Chennai → Bengaluru

or:

    Chennai → Bengaluru → Mysuru

This is graph traversal expressed directly
in the query.

# CELL 43

# Return the Cities in the Path

Instead of returning only the path,
we can return the nodes.

Question:

> Which cities occur on the route
> from Chennai to Mysuru?

In [21]:
# CELL 44
# Instead of returning the entire path graph object,
# it uses the built-in Cypher function nodes(path) to 
# extract an ordered array of the nodes contained within the path, 
# returning [Chennai, Bengaluru, Mysuru].
query = """
MATCH path = (c:City {name: "Chennai"})-[:CONNECTED_TO*1..2]->(m:City {name: "Mysuru"})
RETURN nodes(path) AS route
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        route_nodes = record["route"]
        city_names = [node.get("name", list(node.labels)[0]) for node in route_nodes]
        print("Route cities:", " -> ".join(city_names))

# MATCH path =
#     (c:City {name: "Chennai"})
#     -[:CONNECTED_TO*1..2]->
#     (m:City {name: "Mysuru"})

# RETURN nodes(path) AS route

Route cities: Chennai -> Bengaluru -> Mysuru


# CELL 45

# Multi-Hop Reasoning

We can now see the difference between:

## Direct Query

    Chennai
       ↓
    CONNECTED_TO
       ↓
    Bengaluru

and:

## Multi-Hop Query

    Chennai
       ↓
    CONNECTED_TO
       ↓
    Bengaluru
       ↓
    CONNECTED_TO
       ↓
    Mysuru

The second query requires following
more than one relationship.

This is why graph traversal is important.

# CELL 46

# Query 8 — Affordable Hotels in Heritage Destinations Near Chennai

Now combine everything we have learned.

Question:

> Find hotels costing less than ₹3500
> located in heritage destinations
> near Chennai.

We need:

    Hotel
       ↓ LOCATED_IN
    Destination
       ↓ HAS_CATEGORY
    Heritage

AND:

    Destination
       ↓ NEAR
    Chennai

AND:

    Hotel.price < 3500

In [22]:
# CELL 47
# It traverses across four distinct node types in a single matching chain:

#     Finds a Hotel priced under ₹3,500 (Hotel Heritage at ₹3,000).

#     Verifies it is LOCATED_IN a Destination (Pondicherry).

#     Ensures that Destination HAS_CATEGORY Heritage.

#     Verifies that Destination is NEAR Chennai.
query = """
MATCH (hotel:Hotel)-[:LOCATED_IN]->(destination:Destination)-[:HAS_CATEGORY]->(category:Category {name: "Heritage"}),
      (destination)-[:NEAR]->(chennai:City {name: "Chennai"})
WHERE hotel.price < 3500
RETURN
    hotel.name AS hotel,
    destination.name AS destination,
    hotel.price AS price
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        print(f"Hotel: {record['hotel']} | Destination: {record['destination']} | Price: ₹{record['price']}")

# MATCH (hotel:Hotel)
#       -[:LOCATED_IN]->
#       (destination:Destination)
#       -[:HAS_CATEGORY]->
#       (category:Category {name: "Heritage"}),
#       (destination)-[:NEAR]->
#       (chennai:City {name: "Chennai"})

# WHERE hotel.price < 3500

# RETURN
#     hotel.name AS hotel,
#     destination.name AS destination,
#     hotel.price AS price

Hotel: Hotel Heritage | Destination: Pondicherry | Price: ₹3000


# CELL 48

# Observe the Query

Look at what we just expressed.

The query follows:

    Hotel
      ↓
    Destination
      ↓
    Heritage

and:

    Destination
      ↓
    Chennai

and applies:

    Price < 3500

The question was expressed almost directly
as a graph pattern.

This is one of the major strengths
of graph databases.

# CELL 49

# Compare This with Our Earlier Python Graph

In KG-03 we had to:

1. Build an adjacency structure
2. Write traversal logic
3. Traverse the graph
4. Filter results
5. Combine conditions

In Neo4j:

    MATCH
       ↓
    graph pattern
       ↓
    WHERE
       ↓
    RETURN

The graph database performs
the matching and traversal for us.

This does NOT mean the concept has changed.

Only the implementation has become
more powerful and scalable.

# CELL 50

# Query 9 — Count Hotels

Graph databases can also perform aggregation.

Question:

> How many hotels do we have?

In [23]:
# CELL 51 It scans all nodes with the Hotel label and 
#uses the Cypher aggregation function count(hotel) to count 
#the total number of hotel entities present
#in the database (returning 2: Hotel SeaView and Hotel Heritage).
query = """
MATCH (hotel:Hotel)
RETURN count(hotel) AS number_of_hotels
"""

with driver.session() as session:
    result = session.run(query)
    record = result.single()
    print("Total Number of Hotels:", record["number_of_hotels"])
# MATCH (hotel:Hotel)
# RETURN count(hotel) AS number_of_hotels

Total Number of Hotels: 2


# CELL 52

# Query 10 — Hotels by Destination

Question:

> Which hotels are available in each destination?

We want:

    Destination → Hotels

In [24]:
# CELL 53
# It groups hotels by their target Destination node and 
# uses the Cypher aggregation function collect(hotel.name) 
# to aggregate all hotel names for each destination into a list.

#     Mahabalipuram: ['Hotel SeaView']

#     Pondicherry: ['Hotel Heritage']
query = """
MATCH (hotel:Hotel)-[:LOCATED_IN]->(destination:Destination)
RETURN
    destination.name AS destination,
    collect(hotel.name) AS hotels
"""

with driver.session() as session:
    result = session.run(query)
    for record in result:
        destination = record["destination"]
        hotels = ", ".join(record["hotels"])
        print(f"Destination: {destination} | Hotels: [{hotels}]")
# MATCH (hotel:Hotel)
#       -[:LOCATED_IN]->
#       (destination:Destination)

# RETURN
#     destination.name AS destination,
#     collect(hotel.name) AS hotels

Destination: Mahabalipuram | Hotels: [Hotel SeaView]
Destination: Pondicherry | Hotels: [Hotel Heritage]


In [25]:
   
from pyvis.network import Network
import IPython

# 1. Fetch all nodes and relationships from Neo4j
query = """
MATCH (n)-[r]->(m)
RETURN n.name AS source, 
       labels(n)[0] AS source_label,
       type(r) AS rel_type,
       m.name AS target, 
       labels(m)[0] AS target_label,
       n.price AS source_price,
       m.price AS target_price
"""

# 2. Initialize Pyvis network graph
net = Network(notebook=True, cdn_resources='remote', height="550px", width="100%", directed=True)

# Define color scheme for each node label
color_map = {
    "City": "#4285F4",        # Blue
    "State": "#EA4335",       # Red
    "Destination": "#FBBC05", # Yellow
    "Category": "#34A853",    # Green
    "Hotel": "#8E44AD"        # Purple
}

# 3. Add nodes and edges with colors and tooltips
with driver.session() as session:
    result = session.run(query)
    for record in result:
        src, src_lbl = record["source"], record["source_label"]
        tgt, tgt_lbl = record["target"], record["target_label"]
        rel = record["rel_type"]
        
        # Tooltips (hover details for price if applicable)
        src_title = f"Label: {src_lbl}" + (f"\nPrice: ₹{record['source_price']}" if record['source_price'] else "")
        tgt_title = f"Label: {tgt_lbl}" + (f"\nPrice: ₹{record['target_price']}" if record['target_price'] else "")
        
        # Add source and target nodes
        net.add_node(src, label=f"{src}\n({src_lbl})", color=color_map.get(src_lbl, "#999999"), title=src_title)
        net.add_node(tgt, label=f"{tgt}\n({tgt_lbl})", color=color_map.get(tgt_lbl, "#999999"), title=tgt_title)
        
        # Add directed edge with relationship label
        net.add_edge(src, tgt, title=rel, label=rel)

# 4. Enable physics for smooth node positioning and drag-and-drop
net.toggle_physics(True)

# 5. Write HTML file and display inside an IFrame
net.write_html("knowledge_graph.html")
IPython.display.IFrame(src="knowledge_graph.html", width="70%", height="570px")

In [26]:
import pandas as pd

query = """
MATCH (h:Hotel)-[:LOCATED_IN]->(d:Destination)-[:NEAR]->(c:City)
MATCH (d)-[:HAS_CATEGORY]->(cat:Category)
RETURN h.name AS Hotel, h.price AS Price_INR, d.name AS Destination, cat.name AS Category, c.name AS Near_City
"""

with driver.session() as session:
    result = session.run(query)
    df = pd.DataFrame([record.data() for record in result])

# Display clean DataFrame
display(df)

# Summary statistics
print("\n--- Summary Statistics ---")
print(f"Average Hotel Price: ₹{df['Price_INR'].mean():.2f}")

,Hotel,Price_INR,Destination,Category,Near_City
0,Hotel SeaView,3500,Mahabalipuram,Heritage,Chennai
1,Hotel Heritage,3000,Pondicherry,Heritage,Chennai



--- Summary Statistics ---
Average Hotel Price: ₹3250.00


In [27]:
#dynamically list graph inventory (node counts by label and relationship counts by type):
# Node counts per label
node_count_query = """
MATCH (n)
RETURN labels(n)[0] AS Label, count(n) AS NodeCount
ORDER BY NodeCount DESC
"""

# Relationship counts per type
rel_count_query = """
MATCH ()-[r]->()
RETURN type(r) AS Relationship, count(r) AS RelCount
ORDER BY RelCount DESC
"""

with driver.session() as session:
    nodes_df = pd.DataFrame([r.data() for r in session.run(node_count_query)])
    rels_df = pd.DataFrame([r.data() for r in session.run(rel_count_query)])

print("--- Node Distribution ---")
display(nodes_df)

print("\n--- Relationship Distribution ---")
display(rels_df)

--- Node Distribution ---


,Label,NodeCount
0,City,3
1,Destination,2
2,Hotel,2
3,State,1
4,Category,1



--- Relationship Distribution ---


,Relationship,RelCount
0,LOCATED_IN,3
1,CONNECTED_TO,2
2,NEAR,2
3,HAS_CATEGORY,2


In [28]:
import pandas as pd

# 1. Fetch Hotels with Destination, Category, Near City, and Price
hotel_query = """
MATCH (h:Hotel)-[:LOCATED_IN]->(d:Destination)-[:HAS_CATEGORY]->(cat:Category)
MATCH (d)-[:NEAR]->(c:City)
RETURN 
    h.name AS Hotel,
    h.price AS Price_INR,
    d.name AS Destination,
    cat.name AS Category,
    c.name AS Near_City
"""

# 2. Fetch Inter-City Connectivity Routes
route_query = """
MATCH (c1:City)-[:CONNECTED_TO]->(c2:City)
RETURN 
    c1.name AS From_City,
    c2.name AS To_City
"""

with driver.session() as session:
    # Execute hotel analytical query
    hotel_res = session.run(hotel_query)
    hotels_df = pd.DataFrame([record.data() for record in hotel_res])
    
    # Execute route query
    route_res = session.run(route_query)
    routes_df = pd.DataFrame([record.data() for record in route_res])

# --- Display DataFrames ---
print("=== Travel Destinations & Hotels Summary ===")
display(hotels_df)

print("\n=== City Connectivity Routes ===")
display(routes_df)

# --- Summary Analytics ---
if not hotels_df.empty:
    print("\n=== Quick Statistics ===")
    print(f"Average Hotel Price: ₹{hotels_df['Price_INR'].mean():.2f}")
    print(f"Cheapest Hotel: {hotels_df.loc[hotels_df['Price_INR'].idxmin()]['Hotel']} (₹{hotels_df['Price_INR'].min()})")

=== Travel Destinations & Hotels Summary ===


,Hotel,Price_INR,Destination,Category,Near_City
0,Hotel SeaView,3500,Mahabalipuram,Heritage,Chennai
1,Hotel Heritage,3000,Pondicherry,Heritage,Chennai



=== City Connectivity Routes ===


,From_City,To_City
0,Chennai,Bengaluru
1,Bengaluru,Mysuru



=== Quick Statistics ===
Average Hotel Price: ₹3250.00
Cheapest Hotel: Hotel Heritage (₹3000)


In [29]:
#Always end your notebook with a clean connection close block to free 
#up pool resources in Neo4j AuraDB:
driver.close()
print("Neo4j driver connection cleanly closed.") 

Neo4j driver connection cleanly closed.


# CELL 54

# A More Realistic Travel Question

Suppose the user asks:

> I want to visit a heritage destination
> near Chennai. Show me affordable hotels.

This is no longer a single fact.

It requires:

    1. Identify Chennai
    2. Find nearby destinations
    3. Identify heritage destinations
    4. Find hotels
    5. Check price
    6. Return the result

The graph lets us express
these relationships together.

# CELL 55

# Question → Graph Pattern

This is the reasoning process:

USER QUESTION
       ↓
Identify entities

    Chennai
    Heritage
    Hotel

       ↓
Identify relationships

    NEAR
    HAS_CATEGORY
    LOCATED_IN

       ↓
Construct graph pattern

    Hotel
       ↓
    LOCATED_IN
       ↓
    Destination
       ↓
    HAS_CATEGORY
       ↓
    Heritage

    Destination
       ↓
    NEAR
       ↓
    Chennai

       ↓
Apply price condition

       ↓
Cypher query

# CELL 56

# Important Connection to KG-03

In KG-03 we learned:

    Question
       ↓
    Entities
       ↓
    Relationships
       ↓
    Graph Pattern
       ↓
    Query
       ↓
    Traversal
       ↓
    Answer

Neo4j + Cypher does NOT replace this thinking.

It gives us a powerful language
for expressing the graph pattern.

Therefore:

> Learn graph thinking first.
> Learn Cypher second.

# CELL 57

# Neo4j vs Our Python Graph

| Python Graph | Neo4j |
|---|---|
| Dictionary | Graph database |
| Tuple | Relationship |
| Manual traversal | Cypher pattern matching |
| Manual search | MATCH |
| Manual filtering | WHERE |
| Manual result construction | RETURN |
| Python code | Cypher |
| In-memory | Persistent database |

The conceptual graph remains the same.

# CELL 58

# Neo4j vs Relational Database

We already studied this in KG-05.

For example:

> Find heritage destinations near Chennai
> with hotels below ₹3500.

Relational DB:

    Tables
       ↓
    JOIN
       ↓
    JOIN
       ↓
    WHERE
       ↓
    Result

Neo4j:

    Hotel
       ↓
    LOCATED_IN
       ↓
    Destination
       ↓
    HAS_CATEGORY
       ↓
    Heritage

    Destination
       ↓
    NEAR
       ↓
    Chennai

    +
    price < 3500

# CELL 59

# The Point Is Not "Neo4j Is Better"

We should NOT conclude:

    Neo4j > Relational Database

Instead:

    Structured data
          ↓
    Relational DB is natural

    Relationship-heavy data
          ↓
    Graph DB is natural

    Semantic similarity
          ↓
    Vector Store is natural
 
The right question is:

> What kind of question are we trying to answer?

# CELL 60

# A Three-System Travel Architecture

Our KG-05 conclusion can now be implemented
as a realistic architecture.

                 TRAVEL ASSISTANT
                        │
                        ↓
                USER QUESTION
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
       SQLite         Neo4j        Chroma
          ↓             ↓             ↓
      Structured    Relations      Semantic
        Data         + Paths       Retrieval
          └─────────────┼─────────────┘
                        ↓
                       LLM
                        ↓
                  Travel Answer
                  
 This is a hybrid knowledge architecture.


---

## CELL 61


# Hands-On Exercise 1

Create the following new knowledge:

    Hyderabad is connected to Bengaluru.

    Hyderabad has category Heritage.

    Charminar Hotel is located in Hyderabad.

    Charminar Hotel costs ₹2800 per night.

Represent this knowledge in Neo4j.

Then answer:

> Which heritage destinations are connected
> to Bengaluru?

# CELL 62

# Hands-On Exercise 2

Add:

    Coimbatore is connected to Bengaluru.

    Coimbatore is connected to Chennai.

Now ask:

> Which cities can be reached from Chennai
> through one or two CONNECTED_TO relationships?

Write the Cypher query yourself.

# CELL 63

# Hands-On Exercise 3

Write a Cypher query for:

> Find hotels below ₹3000.

Think first :
What is the entity?

What is the property?

What is the condition?

Then write
MATCH
   ↓
WHERE
   ↓
RETURN


---


# CELL 64

# Hands-On Exercise 4

Write a Cypher query for:

> Find heritage destinations near Chennai.

Think in graph patterns:

    Destination
          ↓
    HAS_CATEGORY
          ↓
       Heritage

and:

    Destination
          ↓
        NEAR
          ↓
       Chennai

# CELL 65

# Hands-On Exercise 5 — Multi-Hop

Write a query for:

> Is there a route from Chennai to Mysuru
> using CONNECTED_TO relationships?

Try to return:

1. The path
2. The cities in the path

Do not look at the earlier solution immediately.

First construct the graph pattern yourself.

# CELL 66

# Reflection

Answer these questions.

### 1.

What is a node in Neo4j?

### 2.

What is a relationship?

### 3.

What is a property?

### 4.

What does MATCH do?

### 5.

What does WHERE do?

### 6.

What does RETURN do?

### 7.

What does *1..2 mean in a relationship pattern?

### 8.

Why is multi-hop traversal natural in a graph database?

### 9.

How is Neo4j different from the Python graph
we built earlier?

### 10.

Why should we not conclude that Neo4j
is always better than a relational database?

# CELL 67

# Final Takeaway

We started with:

    REAL-WORLD TRAVEL PROBLEM

and moved through:

    KNOWLEDGE
       ↓
    TRIPLES
       ↓
    GRAPH
       ↓
    PYTHON IMPLEMENTATION
       ↓
    NEO4J
       ↓
    CYPHER
       ↓
    QUERY
       ↓
    TRAVERSAL
       ↓
    MULTI-HOP REASONING

The important learning is not the syntax alone.

It is:

> A graph database stores relationships
> as first-class elements of the data.

And Cypher allows us to ask:

> "What pattern exists in this graph?"

rather than thinking only in terms of tables and joins.

# CELL 68

# The Big Picture

We have now built the SAME travel knowledge
in several representations.

### Python Graph

    Nodes + Edges
         ↓
    Manual Traversal

### Relational Database

    Tables
         ↓
    SQL

### Vector Store

    Embeddings
         ↓
    Similarity Search

### Neo4j

    Nodes + Relationships
         ↓
    Cypher
         ↓
    Graph Traversal

Each representation has a purpose.

---

## Next Notebook

# KG-07 — RDF and SPARQL

We will now ask:

> Is there another formal way
> to represent Knowledge Graphs?

We will move from:

    Property Graph
         ↓
      Neo4j

to:

    RDF
         ↓
     SPARQL

The travel problem will remain the same.

Students first experience Neo4j Browser + Cypher. They see the graph, execute queries, and understand what the database is doing.
after that, 
Python
   ↓
Neo4j Driver
   ↓
Cypher
   ↓
Neo4j


Neo4j = the graph database engine
Neo4j Browser = the interface through which we talk to the database
Neo4j is the graph database technology.

Aura is the cloud-managed way of running Neo4j.

AuraDB is the managed graph database service within Aura.

Python
   ↓
Application/programming language

Neo4j
   ↓
Graph database

Cypher
   ↓
Query language for Neo4j

Neo4j Browser
   ↓
Interface for interacting with Neo4j

In [30]:
                         Tamil Nadu
                              ↑
                              │ LOCATED_IN
                              │
                           Chennai
                         /         \
                    NEAR           CONNECTED_TO
                    /                 \
          Mahabalipuram            Bengaluru
             /    \                    │
        HAS_CATEGORY  LOCATED_IN        │
           /             \              │
      Heritage       Hotel SeaView      │
                          ₹3500         │
                                       │
                                  CONNECTED_TO
                                       │
                                     Mysuru
                        
                        develop the graph

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 5)

In [ ]:
Each student can create an AuraDB instance and work with it through the browser.

 hands-on :
Create AuraDB
     ↓
Open Neo4j Browser
     ↓
Create nodes
     ↓
Create relationships
     ↓
Visualize graph
     ↓
Write Cypher
     ↓
Query graph
     ↓
Multi-hop traversal
     ↓
Solve travel-planning problems

In [ ]:
Neo4j Aura
   │
   │ hosts your graph database
   │
   ↓
Neo4j Console / Browser
   │
   │ you interact with the database
   │
   ↓
Cypher
   │
   │ queries / creates / modifies
   │
   ↓
Knowledge Graph